In [ ]:
# Done( about 1 hr)
from playsound import playsound
import os
import random
import pandas as pd
import os_toolkit as ost

from pydub import AudioSegment
from pydub.playback import play
from pathlib import Path
import dataframe_short as ds
import dataframe_short.display as dsd
import whisper 

import ffmpeg
from playsound import playsound
from pydub import AudioSegment
from pydub.playback import play
from play_audio_file import play_alarm_done


In [18]:
def format_float(x):
    import numpy as np
    # Handle integer and float types separately
    if isinstance(x, (int, np.int64)):
        return f' {x:,.0f}' # no decimals for integers
    elif isinstance(x, (float, np.float64)):
        if abs(x) >= 1:
            return f' {x:,.0f}' # no decimals for large floats
        elif abs(x) > 0.01:
            return f'{x:,.0%}' # percent for medium floats
        elif x == 0:
            return '0' # special case for zero
        else:
            return f' {x:,.3f}' # three decimals for small floats
    else:
        return x # if not a numeric type, return the value as is

pd.set_option('display.max_rows',500)
pd.set_option('display.max_columns',500)
pd.set_option('display.float_format', format_float)

In [2]:
model_base = whisper.load_model('base')
model_large = whisper.load_model('large')
play_alarm_done()

In [3]:
audio_folder = fr"C:\C_Video_Python\3 Body Problem\3 Body Problem Season 01\Season 01 Splitted Audio\French\3 Body Problem_S01E03_FR_FR"

audio_name_list = ost.get_filename(audio_folder)
audio_full_path_list = ost.get_full_filename(audio_folder)


In [26]:
# Your input list
audio_indices = [117, 135, 136, 138, 139, 150, 158, 159, 160, 163, 164, 166, 167, 169, 171, 176, 180, 190, 191, 195, 199, 212, 215, 223, 226, 227, 232, 252, 276, 298, 301, 305, 306, 307, 308]

# 1. Create lists for names and paths using list comprehensions
# Each item in the result list corresponds to the index in audio_indices
curr_audio_names = [
    next((s for s in audio_name_list if f"_{str(idx).zfill(3)}_" in s), None) 
    for idx in audio_indices
]

curr_audio_full_paths = [
    next((s for s in audio_full_path_list if f"_{str(idx).zfill(3)}_" in s), None) 
    for idx in audio_indices
]

# 2. Extract text from sub for each (with a safety check for None)
texts_from_sub = [
    name.split('_')[-1] if name else "Not Found" 
    for name in curr_audio_names
]

In [27]:
# curr_audio_names[0]
(curr_audio_names[0],curr_audio_full_paths[0],texts_from_sub[0] )

("3 Body Problem FR_S01E03_117_– La fillette est toujours là. – Qu'est–ce qui tourne pas rond .mp3",
 "C:\\C_Video_Python\\3 Body Problem\\3 Body Problem Season 01\\Season 01 Splitted Audio\\French\\3 Body Problem_S01E03_FR_FR\\3 Body Problem FR_S01E03_117_– La fillette est toujours là. – Qu'est–ce qui tourne pas rond .mp3",
 "– La fillette est toujours là. – Qu'est–ce qui tourne pas rond .mp3")

In [29]:
audio_info_dict: dict[int,tuple] = {}
for i in range(len(audio_indices)):
    audio_info_dict[audio_indices[i]] = (curr_audio_names[i],curr_audio_full_paths[i],texts_from_sub[i] )

In [30]:
# curr_audio_names

In [32]:
import time
result = []
start_time = time.time()

for i, curr_audio_info in tqdm(audio_info_dict.items(), desc="Transcribing"):
    text_pred_large = model_large.transcribe(curr_audio_info[1], language="fr")['text']
    result.append(text_pred_large)

end_time = time.time() # Record end time
duration_minutes = (end_time - start_time) / 60
play_alarm_done()

Transcribing:   0%|          | 0/35 [00:00<?, ?it/s]

In [38]:
print(f"Total transcription time: {duration_minutes:.2f} minutes")

Total transcription time: 20.82 minutes


In [39]:
# result

In [40]:
result_df = pd.DataFrame({
    'audio_index': audio_indices,
    'ori_text': texts_from_sub,
    'whisper': result
})

In [41]:
dsd.display_nice_df(result_df)

,audio_index,ori_text,whisper
0,117,– La fillette est toujours là. – Qu'est–ce qui tourne pas rond .mp3,"La petite fille est là à chaque partie. Non, mais qu'est-ce qui tourne pas rond chez vous ?"
1,135,C'est pas pareil.mp3,C'est pas du tout la même chose.
2,136,C'est un peu pareil.mp3,C'est un peu la même chose.
3,138,Ce sont des réalités virtuelles impossibles à distinguer de la réalité.mp3,Ce sont des réalités virtuelles impossibles à distinguer de la réalité actuelle.
4,139,"Alors, qui est derrière tout ça, monsieur le génie .mp3","Très bien, alors, qui est derrière tout ça, monsieur le génie ?"
5,150,"Je vais pas me suicider, c'est ridicule.mp3","Écoutez, je vais pas me suicider, c'est ri-"
6,158,pour sauver la civilisation.mp3,pour sauver la prochaine civilisation.
7,159,On doit trouver comme prévoir la prochaine ère régulière.mp3,Donc il faut qu'on trouve une façon de prévoir la prochaine ère régulière.
8,160,et sa durée.mp3,et combien de temps elle va durer.
9,163,"[rire moqueur] Putain, t'es complètement givrée.mp3",Oh putain t'es complètement chif...


In [42]:
result_df.to_clipboard()